In [2]:
# Data Manipulation and Math
import random
import math
import pandas as pd
import networkx as nx

# System Performance and Benchmarking
import psutil
import time
import os
import json   # Only if exporting summary to JSON

#Visualisation
import matplotlib.pyplot as plt

Implementation of Simulated Annealing. 
The implementation includes:

- init – Initializes the graph and stores the list of vertices.
- cut_value – Computes the cut value of a given partition.
- neighbour – Generates a neighboring solution by flipping the partition assignment of a randomly selected vertex.
- optimal_cut – Computes the optimal cut value using exhaustive search for comparison.
- optimize – Executes the simulated annealing algorithm, applies the cooling schedule and acceptance criterion, tracks the best solution, computes performance metrics, and returns the final results.

In [4]:
class SimulatedAnnealing:

    def __init__(self, graph):

        self.graph = graph
        self.nodes = list(graph.nodes())

    def cut_value(self, partition):

        cut = 0

        for u, v in self.graph.edges():

            if partition[u] != partition[v]:
                cut += 1

        return cut

    def neighbour(self, partition):

        new_partition = partition.copy()

        node = random.choice(self.nodes)

        new_partition[node] = 1 - new_partition[node]

        return new_partition

    def optimal_cut(self):

        n = len(self.nodes)

        best = 0

        for mask in range(1 << n):

            partition = {}

            for i, node in enumerate(self.nodes):
                partition[node] = (mask >> i) & 1

            best = max(best, self.cut_value(partition))

        return best

    def optimize(
        self,
        initial_partition,
        initial_temp=100,
        cooling_rate=0.995,
        min_temp=0.001,
        max_iterations=1000
    ):

        current = initial_partition.copy()
        current_cut = self.cut_value(current)

        best = current.copy()
        best_cut = current_cut

        cut_history = [current_cut]

        temperature = initial_temp

        iteration = 0

        while temperature > min_temp and iteration < max_iterations:

            candidate = self.neighbour(current)
            candidate_cut = self.cut_value(candidate)

            delta = candidate_cut - current_cut

            if delta > 0:

                current = candidate
                current_cut = candidate_cut

            else:

                probability = math.exp(delta / temperature)

                if random.random() < probability:

                    current = candidate
                    current_cut = candidate_cut

            cut_history.append(current_cut)

            if current_cut > best_cut:

                best = current.copy()
                best_cut = current_cut

            temperature *= cooling_rate
            iteration += 1

        average_cut = sum(cut_history) / len(cut_history)

        optimal_cut = self.optimal_cut()

        approximation_ratio = (
            best_cut / optimal_cut
            if optimal_cut > 0
            else 0
        )

        return {

            "best_partition": best,
            "best_cut": best_cut,
            "average_cut": average_cut,
            "approximation_ratio": approximation_ratio

        }

Generates a random d-regular graph, where every node has the same specified degree (default = 3), and returns the generated graph.

In [5]:
def generate_regular_graph(num_nodes, degree=3):

    graph = nx.random_regular_graph(degree, num_nodes)

    return graph

Creates and returns a random initial partition by assigning each vertex to one of two partitions.

In [6]:
def initialize_partition(graph):

    partition = {}

    for node in graph.nodes():
        partition[node] = random.randint(0, 1)

    return partition

Benchmarks the simulated annealing algorithm across different graph sizes and multiple trials, recording :
- execution time
- CPU time
- throughput
- memory usage
- best cut
- average cut
- approximation ratio

and returns the collected performance results.

In [7]:
def benchmark_simulated_annealing(
    graph_sizes,
    trials=10
):

    process = psutil.Process(os.getpid())

    results = []

    for size in graph_sizes:

        print(f"Benchmarking {size} nodes...")

        for trial in range(1, trials + 1):

            graph = generate_regular_graph(size)

            partition = initialize_partition(graph)

            sa = SimulatedAnnealing(graph)

            cpu_start = time.process_time()
            wall_start = time.perf_counter()

            output = sa.optimize(partition)

            wall_end = time.perf_counter()
            cpu_end = time.process_time()

            runtime = wall_end - wall_start
            cpu_time = cpu_end - cpu_start

            throughput = size / runtime

            peak_ram = process.memory_info().rss / (1024 * 1024)

            results.append({

                "Nodes": size,
                "Trial": trial,
                "Runtime (s)": runtime,
                "CPU Time (s)": cpu_time,
                "Throughput": throughput,
                "Peak RAM (MB)": peak_ram,
                "Best Cut": output["best_cut"],
                "Average Cut": output["average_cut"],
                "Approximation Ratio": output["approximation_ratio"]

            })

    return results

Defines the list of graph sizes to be used for benchmarking.

In [9]:
graph_sizes = [4, 6, 8, 10]

results = benchmark_simulated_annealing(
    graph_sizes=graph_sizes,
    trials=10
)

Benchmarking 4 nodes...
Benchmarking 6 nodes...
Benchmarking 8 nodes...
Benchmarking 10 nodes...


Converts the benchmark results into a DataFrame and displays the collected performance metrics in tabular form.

In [10]:
df = pd.DataFrame(results)

display(df)

,Nodes,Trial,Runtime (s),CPU Time (s),Throughput,Peak RAM (MB),Best Cut,Average Cut,Approximation Ratio
0,4,1,0.019521,0.015625,204.910685,157.070312,4,3.273726,1.0
1,4,2,0.009649,0.015625,414.533545,157.070312,4,3.238761,1.0
2,4,3,0.007503,0.000000,533.141405,157.070312,4,3.170829,1.0
3,4,4,0.010901,0.000000,366.948912,157.070312,4,3.183816,1.0
4,4,5,0.011723,0.015625,341.195036,157.070312,4,3.255744,1.0
5,4,6,0.008436,0.015625,474.163991,157.070312,4,3.317682,1.0
6,4,7,0.010711,0.000000,373.444368,157.070312,4,3.217782,1.0
7,4,8,0.011164,0.015625,358.304146,157.070312,4,3.230769,1.0
8,4,9,0.010744,0.015625,372.283495,157.070312,4,3.282717,1.0
9,4,10,0.010616,0.000000,376.775557,157.070312,4,3.195804,1.0


Groups the benchmark results by graph size and computes the mean and standard deviation for all performance metrics.

In [11]:
summary_df = (
    df.groupby("Nodes")
      .agg({
          "Runtime (s)": ["mean", "std"],
          "CPU Time (s)": ["mean", "std"],
          "Throughput": ["mean", "std"],
          "Peak RAM (MB)": ["mean", "std"],
          "Best Cut": ["mean", "std"],
          "Average Cut": ["mean", "std"],
          "Approximation Ratio": ["mean", "std"]
      })
)

summary_df

Runtime (s)           CPU Time (s)            Throughput             \
             mean       std         mean       std        mean        std   
Nodes                                                                       
4        0.011097  0.003232     0.009375  0.008069  381.570114  85.896005   
6        0.015800  0.000691     0.015625  0.007366  380.400231  16.784594   
8        0.021829  0.001156     0.021875  0.008069  367.404984  19.180126   
10       0.043076  0.003248     0.042188  0.010546  233.320875  17.321022   

      Peak RAM (MB)      Best Cut           Average Cut            \
               mean  std     mean       std        mean       std   
Nodes                                                               
4        157.070312  0.0      4.0  0.000000    3.236763  0.046684   
6        157.070312  0.0      7.2  0.632456    5.096404  0.125657   
8        157.070312  0.0     10.4  0.843274    6.842857  0.213406   
10       157.070312  0.0     12.9  0.316228    8.416883  0.145474   

      Approximation Ratio       
                     mean  std  
Nodes                           
4                     1.0  0.0  
6                     1.0  0.0  
8                     1.0  0.0  
10                    1.0  0.0

Saves the detailed benchmark results and summary statistics as CSV files for further analysis and reporting.

In [12]:
df.to_csv(
    "simulated_annealing_results.csv",
    index=False
)

summary_df.to_csv(
    "simulated_annealing_summary.csv"
)

Convert and save the summary results in JSON format.

In [13]:
json_df = summary_df.copy()

json_df.columns = [
    f"{col[0]}_{col[1]}"
    for col in json_df.columns
]

json_df = json_df.reset_index()

# Rename to match the paper terminology
json_df = json_df.rename(columns={
    "best_cut_mean": "mean_best_cut",
    "best_cut_std": "std_best_cut"
})

json_df.to_json(
    "simulated_annealing_summary.json",
    orient="records",
    indent=4
)